In [1]:
import os
import numpy as np
import pandas as pd

from rdkit import Chem, RDLogger
from rdkit.Chem import (
    AllChem,
    Descriptors,
    Lipinski,
    rdMolDescriptors
)

RDLogger.DisableLog('rdApp.*')


# =========================================================
# canonical smiles
# Ensure consistent results for different SMILES of the same molecule
# =========================================================
def get_canonical_mol(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    canonical = Chem.MolToSmiles(mol, canonical=True)

    mol = Chem.MolFromSmiles(canonical)

    return mol


# =========================================================
# Longest Conjugated Path
# =========================================================
def longest_conjugated_path(mol):

    conj_bonds = []

    for bond in mol.GetBonds():

        if bond.GetIsConjugated():

            conj_bonds.append(
                (
                    bond.GetBeginAtomIdx(),
                    bond.GetEndAtomIdx()
                )
            )

    if not conj_bonds:
        return 0

    graph = {}

    for a, b in conj_bonds:

        graph.setdefault(a, []).append(b)
        graph.setdefault(b, []).append(a)

    longest = 0

    def dfs(node, visited):

        nonlocal longest

        longest = max(longest, len(visited))

        for nbr in graph.get(node, []):

            if nbr not in visited:

                dfs(nbr, visited | {nbr})

    for node in graph:

        dfs(node, {node})

    return longest


# =========================================================
# Merge ring systems
# =========================================================
def merge_rings(rings):

    merged = []

    for ring in rings:

        s = set(ring)

        merged_flag = False

        for i, m in enumerate(merged):

            if not s.isdisjoint(m):

                merged[i] = m.union(s)

                merged_flag = True

                break

        if not merged_flag:

            merged.append(s)

    return merged


# =========================================================
# Get bridge atoms
# =========================================================
def get_bridge_atoms(mol, ringA, ringB):

    bridge_atoms = set()

    for atom in mol.GetAtoms():

        idx = atom.GetIdx()

        if idx in ringA or idx in ringB:
            continue

        nbrs = [n.GetIdx() for n in atom.GetNeighbors()]

        connectA = any(n in ringA for n in nbrs)
        connectB = any(n in ringB for n in nbrs)

        if connectA or connectB:
            bridge_atoms.add(idx)

    return sorted(list(bridge_atoms))


# =========================================================
# Ring center
# =========================================================
def ring_center(conf, ring):

    coords = []

    for i in ring:

        pos = conf.GetAtomPosition(i)

        coords.append([pos.x, pos.y, pos.z])

    return np.mean(coords, axis=0)


# =========================================================
# Ring normal vector
# =========================================================
def ring_normal(conf, ring):

    pts = []

    for i in ring:

        pos = conf.GetAtomPosition(i)

        pts.append([pos.x, pos.y, pos.z])

    pts = np.array(pts)

    centroid = pts.mean(axis=0)

    pts -= centroid

    u, s, vh = np.linalg.svd(pts)

    normal = vh[-1]

    return normal / np.linalg.norm(normal)


# =========================================================
# Angle Between Aromatic Ring Planes
# =========================================================
def ring_plane_angle(conf, ringA, ringB):

    try:

        n1 = ring_normal(conf, ringA)

        n2 = ring_normal(conf, ringB)

        cosang = np.clip(np.dot(n1, n2), -1, 1)

        angle = np.degrees(np.arccos(abs(cosang)))

        return angle

    except:

        return 0


# =========================================================
# Bridge features
# =========================================================
def bridge_features(mol):

    feats = {
        "Bridge Atom Count": 0,
        "Bridge Bond Order Sum": 0,
        "Bridge Pi Electron Count": 0,
        "Bridge Rotatable Bond Count": 0,
        "Distance Between Ring Centers": 0,
        "Angle Between Aromatic Ring Planes": 0
    }

    ring_info = mol.GetRingInfo()

    rings = ring_info.AtomRings()

    ring_systems = merge_rings(rings)

    aromatic_systems = []

    for rs in ring_systems:

        if any(
            mol.GetAtomWithIdx(i).GetIsAromatic()
            for i in rs
        ):

            aromatic_systems.append(
                sorted(list(rs))
            )

    if len(aromatic_systems) < 2:
        return feats

    ringA = aromatic_systems[0]
    ringB = aromatic_systems[1]

    bridge_atoms = get_bridge_atoms(
        mol,
        ringA,
        ringB
    )

    bridge_bonds = []

    for b in mol.GetBonds():

        a1 = b.GetBeginAtomIdx()
        a2 = b.GetEndAtomIdx()

        if a1 in bridge_atoms or a2 in bridge_atoms:

            bridge_bonds.append(b)

    # =========================
    # bridge bond order
    # =========================
    bond_sum = sum(
        b.GetBondTypeAsDouble()
        for b in bridge_bonds
    )

    # =========================
    # bridge π electrons
    # =========================
    pi_e = 0

    for b in bridge_bonds:

        t = b.GetBondTypeAsDouble()

        if t == 2:

            pi_e += 2

        elif t == 3:

            pi_e += 4

        elif t == 1.5:

            pi_e += 1

    # =========================
    # bridge rotatable bonds
    # =========================
    rot_bonds = 0

    for b in bridge_bonds:

        if b.IsInRing():
            continue

        if b.GetBondType() != Chem.BondType.SINGLE:
            continue

        a1 = b.GetBeginAtom()
        a2 = b.GetEndAtom()

        if a1.GetDegree() > 1 and a2.GetDegree() > 1:
            rot_bonds += 1

    # =========================
    # 3D geometry
    # =========================
    try:

        mol3d = Chem.AddHs(Chem.Mol(mol))

        params = AllChem.ETKDGv3()

        params.randomSeed = 42

        AllChem.EmbedMolecule(
            mol3d,
            params
        )

        AllChem.MMFFOptimizeMolecule(mol3d)

        conf = mol3d.GetConformer()

        centerA = ring_center(conf, ringA)

        centerB = ring_center(conf, ringB)

        dist = np.linalg.norm(
            centerA - centerB
        )

        angle = ring_plane_angle(
            conf,
            ringA,
            ringB
        )

    except:

        dist = 0
        angle = 0

    feats = {

        "Bridge Atom Count": len(bridge_atoms),

        "Bridge Bond Order Sum": bond_sum,

        "Bridge Pi Electron Count": pi_e,

        "Bridge Rotatable Bond Count": rot_bonds,

        "Distance Between Ring Centers": dist,

        "Angle Between Aromatic Ring Planes": angle
    }

    return feats


# =========================================================
# Main feature function
# =========================================================
def calculate_features(smiles):

    if pd.isna(smiles):
        return {}

    mol = get_canonical_mol(smiles)

    if mol is None:
        return {}

    num_atoms = mol.GetNumAtoms()

    num_bonds = mol.GetNumBonds()

    num_rings = rdMolDescriptors.CalcNumRings(mol)

    num_aromatic_rings = rdMolDescriptors.CalcNumAromaticRings(mol)

    # =========================
    # aromatic carbon
    # =========================
    arom_c = sum(

        1 for a in mol.GetAtoms()

        if a.GetSymbol() == "C"
        and a.GetIsAromatic()
    )

    # =========================
    # hetero atoms
    # =========================
    hetero_atoms = sum(

        1 for a in mol.GetAtoms()

        if a.GetSymbol() in ["N", "O", "S"]
    )

    n_count = sum(

        1 for a in mol.GetAtoms()

        if a.GetSymbol() == "N"
    )

    o_count = sum(

        1 for a in mol.GetAtoms()

        if a.GetSymbol() == "O"
    )

    # =========================
    # π electrons
    # =========================
    pi_electrons = 0

    for b in mol.GetBonds():

        t = b.GetBondTypeAsDouble()

        if t == 2:

            pi_electrons += 2

        elif t == 3:

            pi_electrons += 4

        elif t == 1.5:

            pi_electrons += 1

    pi_density = (

        pi_electrons / num_atoms

        if num_atoms else 0
    )

    # =========================
    # sp3
    # =========================
    sp3_frac = rdMolDescriptors.CalcFractionCSP3(mol)

    # =========================
    # rigidity
    # =========================
    rigid_index = (

        num_rings / num_bonds

        if num_bonds else 0
    )

    # =========================
    # physicochemical
    # =========================
    tpsa = Descriptors.TPSA(mol)

    logp = Descriptors.MolLogP(mol)

    mr = Descriptors.MolMR(mol)

    mw = Descriptors.MolWt(mol)

    # =========================
    # partial charges
    # =========================
    max_partial = Descriptors.MaxPartialCharge(mol)

    min_partial = Descriptors.MinPartialCharge(mol)

    # =========================
    # aromatic bond ratio
    # =========================
    aromatic_bond_ratio = (

        sum(
            1 for b in mol.GetBonds()
            if b.GetIsAromatic()
        )

        / num_bonds

        if num_bonds else 0
    )

    # =========================
    # conjugation
    # =========================
    longest_conj = longest_conjugated_path(mol)

    # =========================
    # Balaban J
    # =========================
    try:

        balabanj = Descriptors.BalabanJ(mol)

    except:

        balabanj = 0

    # =========================
    # bridge features
    # =========================
    bridge_feat = bridge_features(mol)

    # =========================
    # final features
    # =========================
    feature = {

        "Atom Count": num_atoms,

        "Bond Count": num_bonds,

        "Aromatic Ring Count": num_aromatic_rings,

        "Total Ring Count": num_rings,

        "Aromatic Carbon Atom Count": arom_c,

        "Oxygen Atom Count": o_count,

        "Heteroatom Count (N/O/S)": hetero_atoms,

        "Total Pi Electrons": pi_electrons,

        "Pi Electron Density": pi_density,

        "Longest Conjugated Path": longest_conj,

        "sp3 Hybridization Fraction": sp3_frac,

        "Molecular Rigidity Index": rigid_index,

        "TPSA": tpsa,

        "logP": logp,

        "Molar Refractivity": mr,

        "Molecular Weight": mw,

        "Maximum Partial Charge": max_partial,

        "Minimum Partial Charge": min_partial,

        "Aromatic Bond Fraction": aromatic_bond_ratio,

        "Hydrogen Bond Donor Count": Lipinski.NumHDonors(mol),

        "Hydrogen Bond Acceptor Count": Lipinski.NumHAcceptors(mol),

        "Rotatable Bond Count": Descriptors.NumRotatableBonds(mol),

        "BalabanJ": balabanj,

        **bridge_feat
    }

    return feature


# =========================================================
# batch run
# =========================================================
if __name__ == "__main__":

    desktop = os.getcwd()

    def process_file(name, filepath):

        print(f"\nProcessing: {name}")

        df = pd.read_excel(filepath)

        feats = df["SMILES"].apply(
            calculate_features
        )

        feat_df = pd.DataFrame(
            feats.tolist()
        )

        final = pd.concat(
            [
                df[["Index"]].reset_index(drop=True),
                feat_df
            ],
            axis=1
        )

        out = os.path.join(
            desktop,
            "generated_features",
            f"{name}-features.xlsx"
        )

        final.to_excel(
            out,
            index=False
        )

        print("Done:", out)

        print(
            "Number of features:",
            len(feat_df.columns)
        )

    # =========================
    # File 1
    # =========================
    process_file(
        "gs-homo-lumo",
        os.path.join(
            desktop,
            "gs-homo-lumo.xlsx"
        )
    )

    # =========================
    # File 2
    # =========================
    process_file(
        "new_molecules",
        os.path.join(
            desktop,
            "new_molecules.xlsx"
        )
    )

    print("\nAll done")


Processing: gs-homo-lumo
Done: D:\MLcof\github\final\generated_features\gs-homo-lumo-features.xlsx
Number of features: 29

Processing: new_molecules
Done: D:\MLcof\github\final\generated_features\new_molecules-features.xlsx
Number of features: 29

All done
